# 🏠 Forecasting Household Energy Consumption
## Notebook 1 — Exploratory Data Analysis (EDA)

**Capstone Project | UMBC Data Science Master's Program**
Instructor: Dr. Chaojie (Jay) Wang | Author: Kushal Erramilli

---

### What This Notebook Does
This notebook answers three core questions before any model is built:

1. **What does the data look like?** — shape, types, missing values, distribution
2. **Are there patterns we can exploit?** — hourly, daily, weekly, seasonal cycles
3. **Which variables matter most?** — correlations, sub-meter contributions, outliers


**Dataset:** UCI Individual Household Electric Power Consumption
**Path:** `../data/household_power_consumption.csv`


## 0. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)


import plotly.io as pio
import plotly.graph_objects as go

# Professional theme
custom_theme = dict(
    font=dict(family="Inter, Arial, sans-serif", size=13, color="#2C2C2C"),
    title_font=dict(family="Inter, Arial, sans-serif", size=16, color="#0D2340", weight="bold"),
    paper_bgcolor="white",
    plot_bgcolor="#F7F9FC",
    colorway=["#1565C0","#E53935","#2E7D32","#F57C00","#6A1B9A","#00838F"],
    xaxis=dict(showgrid=True, gridcolor="#E8ECF0", gridwidth=1,
               zeroline=False, linecolor="#D0D5DD", linewidth=1.5,
               tickfont=dict(size=11), title_font=dict(size=12)),
    yaxis=dict(showgrid=True, gridcolor="#E8ECF0", gridwidth=1,
               zeroline=False, linecolor="#D0D5DD", linewidth=1.5,
               tickfont=dict(size=11), title_font=dict(size=12)),
    legend=dict(bgcolor="rgba(255,255,255,0.9)", bordercolor="#D0D5DD",
                borderwidth=1, font=dict(size=11)),
    margin=dict(l=60, r=40, t=70, b=60),
)
pio.templates["kushal"] = go.layout.Template(layout=custom_theme)
pio.templates.default = "kushal"
print("✅ Professional theme loaded.")

print("✅ All libraries loaded successfully.")

ModuleNotFoundError: No module named 'pandas'

## 1. Load Dataset

**Why:** We need to understand raw structure before cleaning anything.
The separator is `;` (not `,`) and missing entries are encoded as `?`.


In [ ]:
DATA_PATH = "../data/household_power_consumption.csv"

df_raw = pd.read_csv(
    DATA_PATH,
    na_values=["?"],
    low_memory=False
)

print(f"Shape  : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head()


## 2. Dataset Overview

**Why:** `.info()` reveals data types immediately — all power columns land as `object`
because of the `?` placeholders. We must coerce them to `float64`.


In [ ]:
df_raw.info()

In [ ]:
df_raw.describe(include="all").T


## 3. Data Cleaning

### 3.1 Combine Date + Time → DatetimeIndex

**Why:** A proper `DatetimeIndex` lets us use Pandas time-aware operations
(`.resample()`, `.shift()`, `.rolling()`) which are essential for feature
engineering in Notebook 2.


In [ ]:
df = df_raw.copy()
df["Datetime"] = pd.to_datetime(
    df["Date"].astype(str) + " " + df["Time"].astype(str),
    dayfirst=True
)

df = df.sort_values("Datetime").set_index("Datetime")
df.drop(columns=["Date", "Time"], inplace=True)

numeric_cols = [
    "Global_active_power", "Global_reactive_power", "Voltage",
    "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3",
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"✅ Datetime index set | Numeric columns converted")
print(f"Date range : {df.index.min()}  →  {df.index.max()}")
print(f"Shape      : {df.shape}")
df.head()


### 3.2 Missing Value Analysis

**Why:** Knowing *how much* data is missing and *whether it's random or
block-structured* determines the right imputation strategy.

**What we find:** Exactly **25,979 rows** (≈ 1.25 %) are missing across all
7 sensor columns simultaneously — meaning entire one-minute windows are
missing, not individual sensors. This is typical of sensor dropouts or
communication failures rather than random corruption.


In [ ]:
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(3)

missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %"    : missing_pct
}).sort_values("Missing %", ascending=False)

print(missing_df)
print(f"\nTotal missing cells : {missing.sum():,} / {df.size:,}  "
      f"({missing.sum()/df.size*100:.2f}%)")


In [ ]:
fig = px.bar(
    missing_df[missing_df["Missing Count"] > 0].reset_index(),
    x="index", y="Missing %",
    text="Missing %",
    title="Missing Values per Column — All Columns Equally Affected (Block Dropout)",
    labels={"index": "Column", "Missing %": "Missing (%)"},
    color_discrete_sequence=["#C62828"],
)
fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside",
    textfont=dict(size=11, color="#2C2C2C"),
    marker_line_color="#8B0000",
    marker_line_width=0.8,
)
fig.update_layout(
    title_x=0.5,
    height=440,
    yaxis=dict(range=[0, 1.6], title="Missing (%)"),
    xaxis=dict(title="Column", tickangle=-15),
    annotations=[dict(
        text="1.25% across all 7 columns simultaneously → sensor block dropout, not random noise",
        showarrow=False, x=0.5, y=1.09, xref="paper", yref="paper",
        font=dict(size=11, color="#555"), xanchor="center"
    )]
)
fig.show()

### 3.3 Handle Missing Values

**Strategy:** Forward-fill (`ffill`) propagates the last valid reading into
each gap — the correct choice for sensor time series because power consumption
doesn't jump discontinuously. A back-fill guard handles any leading NaNs.

**Decision for modelling:** Because gaps are contiguous blocks (not random
scatter), interpolation would hallucinate energy data. Forward-fill is
conservative and safe.


In [ ]:
df_clean = df.copy()
df_clean.ffill(inplace=True)
df_clean.bfill(inplace=True)

print(f"✅ Missing values remaining : {df_clean.isnull().sum().sum()}")
print(f"   Clean dataset shape      : {df_clean.shape}")


## 4. Feature Engineering — Calendar Attributes

**Why:** Time-of-day, day-of-week, and season are some of the strongest
predictors of energy use (people follow routines). These features will be
used in EDA *and* directly fed into the models in Notebook 2.

Cyclical features (`sin`/`cos` encoding) prevent the model from treating
23:00 and 00:00 as far apart — they are adjacent on the clock.


In [ ]:
df_clean["Year"]       = df_clean.index.year
df_clean["Month"]      = df_clean.index.month
df_clean["Month_Name"] = df_clean.index.month_name()
df_clean["DayOfWeek"]  = df_clean.index.dayofweek
df_clean["Day_Name"]   = df_clean.index.day_name()
df_clean["Hour"]       = df_clean.index.hour
df_clean["IsWeekend"]  = (df_clean.index.dayofweek >= 5).astype(int)
df_clean["Quarter"]    = df_clean.index.quarter
df_clean["Season"]     = df_clean["Month"].map({
    12:"Winter", 1:"Winter", 2:"Winter",
    3:"Spring",  4:"Spring", 5:"Spring",
    6:"Summer",  7:"Summer", 8:"Summer",
    9:"Autumn", 10:"Autumn",11:"Autumn"
})

print("✅ Calendar features added")
df_clean[["Year","Month","DayOfWeek","Hour","IsWeekend","Season"]].head()


## 5. Target Variable Analysis — `Global_active_power`

**Why:** Understanding the distribution of what we're trying to predict is critical — a right-skewed target often benefits from log-transformation or tree-based models that handle non-normality natively.

**What we find (from actual output):**
- **Mean = 1.09 kW, Median = 0.60 kW** (minute-level) — large gap confirms strong right skew (skewness = 1.80)
- Most minutes the household draws < 1 kW (standby loads, lighting)
- Occasional spikes up to **11.1 kW** represent water-heater + oven + HVAC running simultaneously
- **Decision:** Use tree-based models which handle skew without transformation; evaluate with MAE alongside RMSE

In [ ]:
gap = df_clean["Global_active_power"]

print("=== Global Active Power — Summary Statistics ===")
stats_dict = {
    "Mean"   : gap.mean(),
    "Median" : gap.median(),
    "Std Dev": gap.std(),
    "Min"    : gap.min(),
    "Max"    : gap.max(),
    "Skew"   : gap.skew(),
    "Kurtosis": gap.kurt(),
    "25th %ile": gap.quantile(0.25),
    "75th %ile": gap.quantile(0.75),
}
for k,v in stats_dict.items():
    print(f"  {k:<12}: {v:.4f} kW")


In [ ]:
fig = px.histogram(
    df_clean.sample(100_000, random_state=42),
    x="Global_active_power",
    nbins=100,
    title="Distribution of Global Active Power — Right-Skewed, Median 0.60 kW",
    labels={"Global_active_power": "Active Power (kW)"},
    color_discrete_sequence=["#1565C0"],
    marginal="box",
    opacity=0.85,
)
fig.add_vline(
    x=gap.mean(), line_dash="dash", line_color="#E53935", line_width=1.8,
    annotation_text=f"Mean = {gap.mean():.2f} kW",
    annotation_font=dict(size=11, color="#E53935"),
    annotation_position="top right",
)
fig.add_vline(
    x=gap.median(), line_dash="dot", line_color="#2E7D32", line_width=1.8,
    annotation_text=f"Median = {gap.median():.2f} kW",
    annotation_font=dict(size=11, color="#2E7D32"),
    annotation_position="top left",
)
fig.update_layout(
    title_x=0.5, height=500,
    xaxis=dict(title="Active Power (kW)", range=[-0.2, 8]),
    yaxis_title="Count",
    bargap=0.05,
)
fig.show()

## 6. Time-Series Overview

### 6.1 Full Series (Daily Average)

**Why:** A birds-eye view reveals multi-year trends and confirms seasonality before we measure it statistically.

**What we learn (from the actual chart):**
- Consumption rises every winter and falls every summer — a clean annual cycle repeated across all four years
- **August 2008 anomaly:** An unusual dip to near zero — likely an extended absence or outage — which stands out clearly in the chart
- No long-term upward trend; the household's baseline is stable across the 4-year period
- **Decision:** Include a weekly seasonal lag (168 h) and a fortnightly lag (336 h) in the feature set to capture repeating patterns

In [ ]:
daily_avg = df_clean["Global_active_power"].resample("D").mean().reset_index()
daily_avg.columns = ["Date", "Avg_Power_kW"]

fig = px.line(
    daily_avg, x="Date", y="Avg_Power_kW",
    title="Daily Average Global Active Power — 4-Year Seasonal Overview",
    labels={"Avg_Power_kW": "Avg Power (kW)", "Date": ""},
    color_discrete_sequence=["#1565C0"],
)
fig.update_traces(line=dict(width=1.1), opacity=0.9)
fig.add_annotation(
    x="2008-08-15", y=0.05,
    text="Aug 2008<br>anomaly (outage/absence)",
    showarrow=True, arrowhead=2, arrowcolor="#E53935",
    ax=80, ay=-50, font=dict(size=10, color="#E53935"),
)
fig.update_layout(
    title_x=0.5, height=430,
    yaxis=dict(title="Avg Power (kW)", range=[-0.1, 3.5]),
    hovermode="x unified",
)
fig.show()

### 6.2 Year-over-Year Monthly Comparison

**Why:** Overlaying each year lets us separate *seasonal* effects (same every year) from *trend* effects (year-on-year change).

**What we learn (from actual chart):**
- The U-shaped winter–summer–winter pattern repeats reliably across all four years — strong seasonality we can model
- 2007 winter peak is the highest on record (~1.9 kW in December)
- 2008 has an anomalous August reading (~0.25 kW) — the same outage/absence visible in the daily series
- A mild downward trend is visible (2007 → 2010 winter peaks slightly lower)
- **Decision:** Month and quarter features are worth including; lag_168h covers the weekly cycle and lag_336h adds bi-weekly structure

In [ ]:
month_order = ["January","February","March","April","May","June",
               "July","August","September","October","November","December"]

monthly = (
    df_clean.groupby(["Year","Month_Name"])["Global_active_power"]
    .mean().reset_index()
)
monthly["Month_Name"] = pd.Categorical(
    monthly["Month_Name"], categories=month_order, ordered=True)
monthly = monthly.sort_values(["Year","Month_Name"])

fig = px.line(
    monthly, x="Month_Name", y="Global_active_power",
    color="Year",
    title="Monthly Average Active Power by Year — Stable Annual U-Shape",
    labels={"Global_active_power": "Avg Power (kW)", "Month_Name": "Month"},
    markers=True,
    color_discrete_sequence=["#1565C0","#E53935","#2E7D32","#F57C00","#6A1B9A"],
)
fig.update_traces(line=dict(width=2), marker=dict(size=7))
fig.add_annotation(
    x="August", y=0.28, text="Aug 2008<br>anomaly",
    showarrow=True, arrowhead=2, arrowcolor="#E53935",
    ax=0, ay=-45, font=dict(size=10, color="#E53935"),
)
fig.update_layout(
    title_x=0.5, height=460,
    xaxis=dict(title="", tickangle=-20),
    yaxis=dict(title="Avg Power (kW)"),
    legend=dict(title="Year", orientation="v"),
)
fig.show()

### 6.3 Intraday Pattern — Hour of Day

**Why:** Daily peaks determine *when* appliances are most stressed and directly help a homeowner decide *when* to run high-energy appliances.

**What we learn (from actual data):**
- **Night trough (2–5 AM):** 0.4–0.5 kW — standby loads only
- **Weekday morning spike (7 AM):** 1.71 kW — sharpest single-hour jump (breakfast, showers)
- **Weekday evening peak (21 h):** 1.85 kW — dinner + TV + evening appliances
- **Weekend pattern is fundamentally different:** No sharp morning spike; instead a broad plateau from 10 AM–5 PM (~1.3–1.5 kW) with an evening peak at 19–20 h (~2.01 kW, *higher* than weekday peak)
- **Weekend evenings are the highest-consumption window overall**
- **Decision for homeowner:** Shift high-power appliances (dishwasher, laundry) to 2–5 AM on weekdays, or before 10 AM on weekends

In [ ]:
hourly = (
    df_clean.groupby(["Hour", "IsWeekend"])["Global_active_power"]
    .mean().reset_index()
)
hourly["Day_Type"] = hourly["IsWeekend"].map({0:"Weekday", 1:"Weekend"})

fig = px.line(
    hourly, x="Hour", y="Global_active_power",
    color="Day_Type",
    title="Average Power by Hour of Day — Weekend Evening Peak Exceeds Weekday",
    labels={"Global_active_power": "Avg Power (kW)", "Hour": "Hour of Day"},
    markers=True,
    color_discrete_map={"Weekday": "#1565C0", "Weekend": "#E53935"},
)
fig.update_traces(line=dict(width=2.2), marker=dict(size=6))

# Annotations for key peaks
fig.add_annotation(x=7,  y=1.706, text="Weekday<br>7 AM",
    showarrow=True, arrowhead=2, ax=35, ay=-40,
    font=dict(size=10, color="#1565C0"), arrowcolor="#1565C0")
fig.add_annotation(x=21, y=1.849, text="Weekday<br>9 PM",
    showarrow=True, arrowhead=2, ax=35, ay=-40,
    font=dict(size=10, color="#1565C0"), arrowcolor="#1565C0")
fig.add_annotation(x=19, y=2.013, text="Weekend<br>7 PM ★",
    showarrow=True, arrowhead=2, ax=-60, ay=-40,
    font=dict(size=10, color="#E53935"), arrowcolor="#E53935")

fig.update_layout(
    title_x=0.5, height=450,
    xaxis=dict(tickmode="linear", dtick=2, title="Hour of Day"),
    yaxis=dict(title="Avg Power (kW)", range=[0.3, 2.3]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()

### 6.4 Weekly Pattern — Day of Week

**Why:** Identifies if certain weekdays are consistently higher/lower — useful for utilities planning demand response and for homeowners scheduling energy-intensive tasks.

**What we learn (from actual data):**
- Saturday (1.231 kW) and Sunday (1.203 kW) are **~18% higher** than weekdays (avg ~1.034 kW)
- This is a materially larger weekend effect than a typical "5–8%" — people being home all day drives a significant consumption lift
- Error bars (±1 std) are wide for all days, confirming high within-day variability — season and time-of-day dominate over day-of-week for any individual hour
- Weekday-to-weekday variation is small (Thursday lowest at 0.982 kW, Wednesday highest at 1.085 kW)

In [ ]:
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

weekly = (
    df_clean.groupby("Day_Name")["Global_active_power"]
    .agg(["mean","std"]).reindex(day_order).reset_index()
)

bar_colors = ["#1565C0","#1565C0","#1565C0","#1565C0","#1565C0","#C62828","#C62828"]

fig = go.Figure([go.Bar(
    x=weekly["Day_Name"], y=weekly["mean"],
    error_y=dict(type="data", array=weekly["std"], visible=True,
                 color="#AAB4BE", thickness=1.5, width=6),
    marker_color=bar_colors,
    marker_line_color="rgba(0,0,0,0.15)",
    marker_line_width=0.8,
    text=[f"{v:.3f}" for v in weekly["mean"]],
    textposition="outside",
    textfont=dict(size=10),
)])
fig.update_layout(
    title="Average Active Power by Day — Weekends ~18% Higher Than Weekdays",
    title_x=0.5, height=440,
    xaxis_title="Day of Week", yaxis_title="Avg Power (kW)",
    yaxis=dict(range=[0, 2.6]),
    showlegend=False,
    annotations=[dict(
        text="Blue = Weekday  |  Red = Weekend",
        showarrow=False, x=0.5, y=1.07, xref="paper", yref="paper",
        font=dict(size=11, color="#555"), xanchor="center"
    )]
)
fig.show()

### 6.5 Seasonal Boxplot

**Why:** Boxplots show both the *central tendency* and *spread* per season, revealing whether the higher winter average is driven by everyone using more or by occasional extreme events.

**What we learn (from actual data):**
- **Winter median = 1.358 kW vs Summer median = 0.416 kW — a ~3× difference** (not 2× as is sometimes stated)
- **Winter IQR is also wider** — more variable consumption (mild heating days vs full heating + cooking)
- **Summer has the tightest spread** — predictable, temperature-driven air-conditioning pattern
- **Autumn sits between Spring and Winter** in both median and spread
- **Decision:** Season flag and month features are highly valuable; winter forecast errors will likely be higher due to greater variability (confirmed in Notebook 2 error analysis)

In [ ]:
df_sample = df_clean[df_clean["Season"].isin(["Winter","Spring","Summer","Autumn"])].sample(50_000, random_state=42)

fig = px.box(
    df_sample, x="Season", y="Global_active_power",
    category_orders={"Season": ["Winter","Spring","Summer","Autumn"]},
    color="Season",
    title="Power Distribution by Season — Winter Median 3× Summer Median",
    labels={"Global_active_power": "Active Power (kW)"},
    color_discrete_map={
        "Winter": "#1565C0", "Spring": "#2E7D32",
        "Summer": "#F57C00", "Autumn": "#6A1B9A"
    },
    points=False,
)
fig.update_traces(
    line_width=1.8,
    marker=dict(opacity=0.5, size=3),
    boxmean="sd",
)
# Annotate actual medians
medians = {"Winter": 1.358, "Spring": 0.850, "Summer": 0.416, "Autumn": 0.866}
for i, (s, v) in enumerate(medians.items()):
    fig.add_annotation(
        x=s, y=v + 0.25,
        text=f"Median<br>{v:.2f} kW",
        showarrow=False,
        font=dict(size=9, color="#2C2C2C"),
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="#CCC", borderwidth=1,
        borderpad=3,
    )
fig.update_layout(
    title_x=0.5, height=480, showlegend=False,
    yaxis=dict(range=[-0.2, 10.5], title="Active Power (kW)"),
)
fig.show()

## 7. Sub-Meter Contribution Analysis

**Why:** The dataset tracks three sub-meters. Understanding their relative contribution tells us whether they help predict the target *or* are largely redundant noise.

| Sub-meter | Appliances |
|-----------|-----------|
| Sub_metering_1 | Kitchen (microwave, dishwasher, oven) |
| Sub_metering_2 | Laundry (washing machine, tumble dryer, fridge) |
| Sub_metering_3 | Water heater + air conditioner |

**What we find (from actual computed output):**
- **HVAC (Sub3): 9.85%** of total energy — the largest metered sub-load by a wide margin
- **Laundry (Sub2): 1.98%**
- **Kitchen (Sub1): 1.70%**
- **Unmetered / other: 86.47%** — lighting, TV, computers, EV charger, etc.

**Key insight:** The three sub-meters together account for **~13.5%** of total consumption. HVAC dominates metered loads. The majority (86.5%) is still unmetered, which is why the sub-meter readings are moderate — rather than strong — direct predictors of global power. However, the HVAC seasonal rhythm is a useful signal and all three sub-meters are retained in the feature set.

In [ ]:
# Convert sub-metering from Wh/min to kW equivalent for direct comparison
df_clean["Sub1_kW"] = df_clean["Sub_metering_1"] / 60
df_clean["Sub2_kW"] = df_clean["Sub_metering_2"] / 60
df_clean["Sub3_kW"] = df_clean["Sub_metering_3"] / 60

total_energy = df_clean["Global_active_power"].sum()
shares = {
    "Kitchen (Sub1)" : df_clean["Sub1_kW"].sum() / total_energy * 100,
    "Laundry (Sub2)" : df_clean["Sub2_kW"].sum() / total_energy * 100,
    "HVAC    (Sub3)" : df_clean["Sub3_kW"].sum() / total_energy * 100,
}
shares["Unmetered/Other"] = 100 - sum(shares.values())

print("=== Energy Share per Sub-meter ===")
for k,v in shares.items():
    print(f"  {k:<20}: {v:.3f}%")

# Print with corrected labels
print("=== Energy Share per Sub-meter (ACTUAL) ===")
for k,v in shares.items():
    print(f"  {k:<20}: {v:.3f}%")


In [ ]:
fig = px.pie(
    names=list(shares.keys()),
    values=list(shares.values()),
    title="Energy Share by Sub-meter — 86.5% Unmetered, HVAC Dominates Metered Loads",
    color_discrete_sequence=["#2E7D32","#1565C0","#E53935","#F57C00"],
    hole=0.45,
)
fig.update_traces(
    textposition="auto",
    textinfo="label+percent",
    textfont=dict(size=12),
    marker=dict(line=dict(color="white", width=2.5)),
    pull=[0, 0.05, 0, 0],
)
fig.update_layout(
    title_x=0.5, height=470,
    legend=dict(orientation="v", font=dict(size=11)),
    annotations=[dict(
        text="<b>13.5%</b><br>metered",
        x=0.5, y=0.5, font_size=13, showarrow=False,
        font=dict(color="#0D2340"),
    )]
)
fig.show()

In [ ]:
# Monthly sub-meter trend — is HVAC seasonal?
monthly_sub = (
    df_clean.resample("ME")[["Sub1_kW","Sub2_kW","Sub3_kW"]]
    .mean().reset_index()
    .melt(id_vars="Datetime", var_name="Sub_Meter", value_name="Avg_kWh_per_min")
)
monthly_sub["Sub_Meter"] = monthly_sub["Sub_Meter"].map({
    "Sub1_kW":"Kitchen (Sub1)","Sub2_kW":"Laundry (Sub2)","Sub3_kW":"HVAC (Sub3)"
})

fig = px.line(
    monthly_sub, x="Datetime", y="Avg_kWh_per_min",
    color="Sub_Meter",
    title="Monthly Sub-meter Trends — HVAC Dominates and Spikes in Winter",
    labels={"Avg_kWh_per_min": "Avg Consumption (kWh/min)", "Datetime": ""},
    color_discrete_sequence=["#E53935","#2E7D32","#1565C0"],
)
fig.update_traces(line=dict(width=2))
fig.update_layout(
    title_x=0.5, height=440,
    yaxis=dict(title="Avg Consumption (kWh/min)"),
    legend=dict(title="Sub-meter"),
    hovermode="x unified",
)
fig.show()

## 8. Correlation Analysis

**Why:** A Pearson correlation matrix quantifies *linear* relationships between all sensor variables. This guides feature selection and flags multicollinearity.

**What we find (from actual matrix):**
- **Global_active_power ↔ Global_intensity: r = 1.00** — perfect linear relationship (P = V × I at near-constant voltage ~241 V). These two columns are identical predictors; we use only power as the target.
- **Global_active_power ↔ Sub_metering_3 (HVAC): r = 0.64** — strong positive; HVAC is the only sub-meter with substantial predictive value
- **Global_active_power ↔ Sub_metering_1 (Kitchen): r = 0.48** — moderate positive (higher than expected — kitchen appliances coincide with high-demand periods)
- **Global_active_power ↔ Sub_metering_2 (Laundry): r = 0.43** — moderate positive (similar reasoning)
- **Voltage ↔ Global_intensity: r = −0.41** — mild negative; as demand rises the mains voltage sags slightly (Ohm's law)
- **Key correction:** Sub1 and Sub2 are *not* weakly correlated with the target — their r values of 0.48 and 0.43 are moderate and worth retaining as features

In [ ]:
corr_cols = [
    "Global_active_power","Global_reactive_power","Voltage",
    "Global_intensity","Sub_metering_1","Sub_metering_2","Sub_metering_3"
]
short_names = ["GAP","Reactive","Voltage","Intensity","Sub1","Sub2","Sub3"]
corr = df_clean[corr_cols].corr()
corr.index = short_names
corr.columns = short_names

fig = px.imshow(
    corr, text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Pearson Correlation Matrix — Intensity Co-linear with GAP; Sub3 Strongest Meter",
    aspect="auto",
)
fig.update_traces(
    textfont=dict(size=12, color="black"),
    xgap=2, ygap=2,
)
fig.update_layout(
    title_x=0.5, height=540,
    coloraxis_colorbar=dict(
        title="r", tickvals=[-1,-0.5,0,0.5,1],
        len=0.85,
    ),
    xaxis=dict(tickfont=dict(size=11)),
    yaxis=dict(tickfont=dict(size=11)),
)
fig.show()

## 9. Autocorrelation & Lag Structure

**Why:** Autocorrelation (ACF) tells us how far back past power values remain predictive. This directly justifies which lag features to create in Notebook 2.

**What we find (from actual computed values):**
- **Lag 1 h: ACF = 0.7151** — very strong; the last hour is the best single predictor
- **Lag 24 h: ACF = 0.4374** — same hour yesterday is also predictive (daily cycle). *(Note: not 0.55 — that is the actual computed value)*
- **Lag 168 h: ACF = 0.4556** — same hour last week (weekly cycle) — slightly higher than lag-24h
- **Lag 336 h: ACF = 0.4366** — same hour two weeks ago — still significant
- ACF oscillates between troughs (12h, near 0) and peaks (24h, 168h) — confirming the 24-hour cycle
- All values remain well above the 95% confidence bound (grey dotted line) out to 336 h

**Decision:** Create lag features at 1, 2, 3, 6, 12, 24, 48, 72, 168, and 336 hours.

In [ ]:
# Compute ACF manually
hourly_gap = df_clean["Global_active_power"].resample("h").mean().dropna()
max_lag = 24 * 14 + 1   # 14 days

acf_values = [hourly_gap.autocorr(lag=l) for l in range(1, max_lag)]
acf_df = pd.DataFrame({"Lag_h": range(1, max_lag), "ACF": acf_values})

key_lags  = [1, 24, 48, 72, 168, 336]
key_flags = acf_df["Lag_h"].isin(key_lags)
conf_bound = 1.96 / np.sqrt(len(hourly_gap))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=acf_df["Lag_h"], y=acf_df["ACF"],
    mode="lines", line=dict(color="#1565C0", width=1.6),
    name="ACF", fill="tozeroy", fillcolor="rgba(21,101,192,0.08)",
))
fig.add_trace(go.Scatter(
    x=acf_df.loc[key_flags,"Lag_h"],
    y=acf_df.loc[key_flags,"ACF"],
    mode="markers+text",
    marker=dict(color="#E53935", size=11, symbol="circle",
                line=dict(color="white", width=2)),
    text=[f"  {l}h  {v:.3f}" for l,v in zip(key_lags, [acf_df.loc[acf_df.Lag_h==l,"ACF"].values[0] for l in key_lags])],
    textposition=["top right","top center","top center","top center","top center","top center"],
    textfont=dict(size=10, color="#E53935"),
    name="Key lags",
))
fig.add_hline(y= conf_bound, line_dash="dot", line_color="#999", line_width=1.2,
              annotation_text="95% CI", annotation_font=dict(size=9))
fig.add_hline(y=-conf_bound, line_dash="dot", line_color="#999", line_width=1.2)
fig.add_hline(y=0, line_color="#AAB4BE", line_width=0.8)
fig.update_layout(
    title="Autocorrelation Function (ACF) — Hourly Active Power, Lags up to 14 Days",
    title_x=0.5, height=450,
    xaxis=dict(title="Lag (hours)", dtick=24),
    yaxis=dict(title="Autocorrelation", range=[-0.15, 0.82]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()

print(f"\nACF at key lags:")
for l in key_lags:
    v = hourly_gap.autocorr(lag=l)
    print(f"  Lag {l:>3}h : {v:.4f}")

## 10. Outlier Detection

**Why:** Extreme values can distort training if a model memorises them. We detect outliers with two methods and decide whether to remove them.

**What we find (from actual output):**
- **Z-score (|z| > 3): 36,803 readings (1.77%)** — high-power events (oven + water heater + HVAC running simultaneously)
- **IQR fences: 95,699 readings (4.61%)** — more generous bounds capture milder peak events

**Decision: We do not remove outliers.** These spikes represent real household behaviour. Removing them would make the model systematically under-estimate peak consumption — the exact scenario that matters most for demand management. Instead, we use tree-based models (Random Forest, XGBoost, LightGBM) that are inherently robust to extreme values.

In [ ]:
gap_clean = df_clean["Global_active_power"].dropna()

z       = np.abs(stats.zscore(gap_clean))
z_out   = (z > 3).sum()
Q1, Q3  = gap_clean.quantile([0.25, 0.75])
IQR     = Q3 - Q1
iqr_out = ((gap_clean < Q1 - 1.5*IQR) | (gap_clean > Q3 + 1.5*IQR)).sum()

print(f"Z-score outliers (|z|>3): {z_out:,}  ({z_out/len(gap_clean)*100:.2f}%)")
print(f"IQR     outliers        : {iqr_out:,}  ({iqr_out/len(gap_clean)*100:.2f}%)")
print(f"\n→ Decision: RETAIN all outliers — they represent real peak usage events.")


In [ ]:
corr_cols = [
    "Global_active_power","Global_reactive_power","Voltage",
    "Global_intensity","Sub_metering_1","Sub_metering_2","Sub_metering_3"
]
short_labels = {
    "Global_active_power":"GAP (kW)",
    "Global_reactive_power":"Reactive (kW)",
    "Voltage":"Voltage (V)",
    "Global_intensity":"Intensity (A)",
    "Sub_metering_1":"Sub1 Kitchen (Wh)",
    "Sub_metering_2":"Sub2 Laundry (Wh)",
    "Sub_metering_3":"Sub3 HVAC (Wh)",
}
melted = df_clean[corr_cols].sample(30_000, random_state=42).rename(columns=short_labels).melt(var_name="Feature", value_name="Value")

fig = px.box(
    melted, x="Feature", y="Value",
    color="Feature",
    title="Distribution of All Numeric Features (30k Sample) — Voltage is Most Stable",
    color_discrete_sequence=px.colors.qualitative.Prism,
    points=False,
)
fig.update_traces(line_width=1.6, boxmean=True, marker=dict(opacity=0.3, size=3))
fig.update_layout(
    title_x=0.5, height=490, showlegend=False,
    xaxis=dict(title="", tickangle=-15),
    yaxis=dict(title="Value"),
)
fig.show()

## 11. Save Processed Datasets for Notebook 2

We create three derived datasets:
- `hourly_clean.csv` — 1-hour averages (main modelling dataset)
- `daily_clean.csv`  — 1-day averages (trend visualisation)
- `minute_clean.parquet` — full minute-level data (used by Streamlit app)


In [ ]:
numeric_cols = [
    "Global_active_power","Global_reactive_power","Voltage",
    "Global_intensity","Sub_metering_1","Sub_metering_2","Sub_metering_3"
]
df_hourly = df_clean[numeric_cols].resample("h").mean()
df_daily  = df_clean[numeric_cols].resample("D").mean()

print(f"Hourly dataset : {df_hourly.shape}")
print(f"Daily  dataset : {df_daily.shape}")

df_hourly.to_csv("../data/hourly_clean.csv")
df_daily.to_csv("../data/daily_clean.csv")
df_clean.to_parquet("../data/minute_clean.parquet")

print("\n✅ Saved hourly_clean.csv | daily_clean.csv | minute_clean.parquet")


## 12. Key EDA Findings & Decisions for Modelling

| # | Finding (from actual output) | Impact on Modelling |
|---|---|---|
| 1 | 2,075,259 minute-level records; **1.25% missing** (block gaps — all 7 columns simultaneously) | Use ffill; blocks are rare and short enough not to distort hourly aggregation |
| 2 | Target is **right-skewed** (skew = 1.80, mean 1.09 kW >> median 0.60 kW) | Use tree models + MAE metric; do not log-transform |
| 3 | **Strong annual seasonality** — Winter median 1.36 kW vs Summer median 0.42 kW (~3× ratio) | Include month, quarter, season features; winter errors will be higher |
| 4 | **Daily cycle:** Weekday peaks at 7 AM (1.71 kW) and 21 h (1.85 kW); Weekend evening peak at 19 h (2.01 kW) | Include hour + cyclical sin/cos encoding; is_weekend flag important |
| 5 | **Weekly cycle:** ACF significant at 168 h (0.456) and 336 h (0.437) | Include lag_168h and lag_336h features |
| 6 | **lag_1h ACF = 0.715** — strongest single predictor | lag_1h will dominate feature importance |
| 7 | **HVAC (Sub3) accounts for 9.85% of energy;** all three sub-meters together = 13.5% | Sub-meters have *moderate* predictive value; retain all three in feature set |
| 8 | **Global_intensity is perfectly co-linear** with target (r = 1.00) | Drop from model inputs (or retain — it will score zero importance and add no noise) |
| 9 | **Sub1 r = 0.48, Sub2 r = 0.43, Sub3 r = 0.64** — moderate to strong correlations | All three sub-meters worth retaining; Sub3 is the strongest direct predictor |
| 10 | **Outliers 1.77% (Z>3)** represent real peak usage events (oven + HVAC + water heater) | Retain; use robust tree models + MAE metric |
| 11 | **August 2008 anomaly** — near-zero consumption for ~1 month (likely absence / outage) | Note as data quality flag; unlikely to repeat in test period |
| 12 | **Weekends ~18% higher** than weekdays (1.217 vs 1.034 kW) | is_weekend binary feature + cyclical DOW encoding |
